<a href="https://colab.research.google.com/github/akmeir07/recipe-data-analysis/blob/main/recipe_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Link scraping

In [ ]:
!pip install lxml

In [ ]:
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}

all_links = []
for i in range(1,1050):
    response = requests.get(f'https://eda.rambler.ru/recepty?page={i}',headers = headers)
    soup = BeautifulSoup(response.text, 'lxml')
    r_links = [r.get('href') for r in soup.select('div.css-1j5xcrd a')]
    all_links.extend(r_links)
    time.sleep(np.random.random())
    print(f'{i}s page scraping done!')
print('Scraping names is all done!')

In [ ]:
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}

all_links = []
for i in range(713,715):
    response = requests.get(f'https://eda.rambler.ru/recepty?page={i}',headers = headers)
    soup = BeautifulSoup(response.text, 'lxml')
    r_links = [r.get('href') for r in soup.select('div.css-1j5xcrd a')]
    all_links.extend(r_links)
    time.sleep(np.random.random())
    print(f'{i}s page scraping done!')
print('Scraping names is all done!')

In [ ]:
import pandas as pd
s = pd.read_csv('undone.csv')

In [ ]:
s1 = pd.Series(all_links)
s1.to_csv('undone2.csv')

In [ ]:
recipe_link = pd.concat([s,s1],ignore_index=True)
recipe_link.to_csv('recipe_link.csv')

In [ ]:
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
}

In [ ]:
print(len(response.text))

In [ ]:
len(all_links)

# Scraping each recipe

In [ ]:
all_links = pd.read_csv('recipe_link.csv')['0'].to_list()

In [ ]:
df.to_csv('recipe_df.csv')

In [ ]:
import json
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

base_url = 'https://eda.rambler.ru'
all_recipes_data = []

headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'
}

cleaned_all_links = [str(url) for url in all_links if isinstance(url, str) or (isinstance(url, float) and not pd.isna(url))]
total_links = len(cleaned_all_links)

print(f"Parsing {total_links} recepies...")

for index, url in enumerate(cleaned_all_links, start=1):
    full_url = base_url + url if not url.startswith('http') else url

    try:
        response = None
        for attempt in range(2):
            try:
                response = requests.get(full_url, headers=headers, timeout=25)
                response.raise_for_status()
                break
            except (requests.exceptions.RequestException, requests.exceptions.Timeout):
                if attempt == 0:
                    time.sleep(3)
                    continue
                else:
                    raise

        if not response: continue

        soup = BeautifulSoup(response.text, 'lxml')
        scripts = soup.find_all('script')
        recipe_json = None

        for s in scripts:
            if s.string and '"pageProps"' in s.string and '"recipe"' in s.string:
                try:
                    recipe_json = json.loads(s.string)
                    break
                except json.JSONDecodeError:
                    continue

        if not recipe_json:
            continue

        recipe = recipe_json.get('props', {}).get('pageProps', {}).get('recipe', {})
        if not recipe: continue

        desc = recipe.get('description') or ""
        anon = recipe.get('announcement') or ""


        steps_list = [step.get('description') for step in recipe.get('recipeSteps', []) if step.get('description')]
        steps_text = " ".join(steps_list)

        ing_notes = []
        for item in recipe.get('composition', []):
            widget = item.get('ingredient', {}).get('widgetData', {})
            raw_note = widget.get('description')
            if raw_note:
                clean_note = BeautifulSoup(raw_note, "lxml").get_text()
                ing_notes.append(clean_note)
        ing_notes_text = " ".join(ing_notes)

        full_text_corpus = f"{desc} {anon} {steps_text} {ing_notes_text}".strip()

        ingredients_raw = []
        for item in recipe.get('composition', []):
            ing_data = item.get('ingredient') or {}
            name = ing_data.get('name') or 'Неизвестно'
            amount = item.get('amount') if item.get('amount') is not None else ''
            unit = (item.get('measureUnit') or {}).get('name') or ''
            ingredients_raw.append(f"{name} ({amount} {unit})")

        ingredients_string = ", ".join(ingredients_raw)


        nutrition = recipe.get('nutritionInfo') or {}

        current_recipe = {
            "name": recipe.get('name'),
            "description": desc if desc else anon,
            "nlp_text": full_text_corpus,
            "ingredients": ingredients_string,
            "portions": recipe.get('portionsCount'),
            "likes": recipe.get('likes'),
            "dislikes": recipe.get('dislikes'),
            "saved_count": recipe.get('inCookbookCount'),
            "calories": nutrition.get('kilocalories'),
            "proteins": nutrition.get('proteins'),
            "fat": nutrition.get('fats'),
            "carbohydrates": nutrition.get('carbohydrates'),
            "duration_mins": recipe.get('cookingTime'),
            "cuisine": (recipe.get('cuisine') or {}).get('name'),
            "url": full_url
        }

        all_recipes_data.append(current_recipe)

        print(f"[{index}/{total_links}] Успешно: {current_recipe['name']}")

        if index % 500 == 0:
            pd.DataFrame(all_recipes_data).to_csv('recipes_progress.csv', index=False, encoding='utf-16')
            print(f"--- Бэкап сохранен на {index} рецептах ---")

    except Exception as e:
        print(f"[{index}/{total_links}] Ошибка на {full_url}: {e}")


df = pd.DataFrame(all_recipes_data)
df.to_csv('eda_full_dataset.csv', index=False, encoding='utf-16')
print("\nРАБОТА ЗАВЕРШЕНА!")
print(f"Всего собрано: {len(df)} рецептов.")

In [ ]:
df = pd.DataFrame(all_recipes_data)
df.to_csv('eda_full_dataset.csv', index=False, encoding='utf-16')
print("\nРАБОТА ЗАВЕРШЕНА!")

In [ ]:
import pandas as pd

eda_df = pd.read_csv('eda_full_dataset.csv', encoding='utf-16')
eda_df.duplicated(keep = False).sum()

In [ ]:
eda_df.shape

In [ ]:
eda_df.head()

In [ ]:
import json
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

manually_adding_links = ['https://eda.rambler.ru/recepty/napitki/imbirnij-napitok-29230','https://eda.rambler.ru/recepty/pasta-picca/zapechenaja-pasta-s-sirom-vetchinoj-goroshkom-23522','https://eda.rambler.ru/recepty/napitki/gustoj-gorjachij-shokolad-32609','https://eda.rambler.ru/recepty/vypechka-deserty/panna-kotta-s-mindalnim-sousom-29212','https://eda.rambler.ru/recepty/zakuski/tushenaja-kapusta-s-kartofelem-14605']

base_url = 'https://eda.rambler.ru'
manually_added_recipes_data = []

headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'
}

total_manual_links = len(manually_adding_links)
print(f"Начинаем парсинг {total_manual_links} вручную добавленных рецептов...")

for index, url in enumerate(manually_adding_links, start=1):
    full_url = url # These are already full URLs

    try:
        response = None
        for attempt in range(2):
            try:
                response = requests.get(full_url, headers=headers, timeout=25)
                response.raise_for_status()
                break
            except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
                if attempt == 0:
                    time.sleep(3)
                    continue
                else:
                    print(f"[{index}/{total_manual_links}] Ошибка при запросе {full_url} после повторной попытки: {e}")
                    raise

        if not response: continue

        soup = BeautifulSoup(response.text, 'lxml')
        scripts = soup.find_all('script')
        recipe_json = None

        for s in scripts:
            if s.string and '"pageProps"' in s.string and '"recipe"' in s.string:
                try:
                    recipe_json = json.loads(s.string)
                    break
                except json.JSONDecodeError:
                    continue

        if not recipe_json:
            print(f"[{index}/{total_manual_links}] JSON не найден на: {full_url}")
            continue

        recipe = recipe_json.get('props', {}).get('pageProps', {}).get('recipe', {})
        if not recipe: continue

        desc = recipe.get('description') or ""
        anon = recipe.get('announcement') or ""

        steps_list = [step.get('description') for step in recipe.get('recipeSteps', []) if step.get('description')]
        steps_text = " ".join(steps_list)

        ing_notes = []
        for item in recipe.get('composition', []):
            widget = item.get('ingredient', {}).get('widgetData', {})
            raw_note = widget.get('description')
            if raw_note:
                clean_note = BeautifulSoup(raw_note, "lxml").get_text()
                ing_notes.append(clean_note)
        ing_notes_text = " ".join(ing_notes)

        full_text_corpus = f"{desc} {anon} {steps_text} {ing_notes_text}".strip()

        ingredients_raw = []
        for item in recipe.get('composition', []):
            ing_data = item.get('ingredient') or {}
            name = ing_data.get('name') or 'Неизвестно'
            amount = item.get('amount') if item.get('amount') is not None else ''
            unit = (item.get('measureUnit') or {}).get('name') or ''
            ingredients_raw.append(f"{name} ({amount} {unit})")

        ingredients_string = ", ".join(ingredients_raw)

        nutrition = recipe.get('nutritionInfo') or {}

        current_recipe = {
            "name": recipe.get('name'),
            "description": desc if desc else anon,
            "nlp_text": full_text_corpus,
            "ingredients": ingredients_string,
            "portions": recipe.get('portionsCount'),
            "likes": recipe.get('likes'),
            "dislikes": recipe.get('dislikes'),
            "saved_count": recipe.get('inCookbookCount'),
            "calories": nutrition.get('kilocalories'),
            "proteins": nutrition.get('proteins'),
            "fat": nutrition.get('fats'),
            "carbohydrates": nutrition.get('carbohydrates'),
            "duration_mins": recipe.get('cookingTime'),
            "cuisine": (recipe.get('cuisine') or {}).get('name'),
            "url": full_url
        }

        manually_added_recipes_data.append(current_recipe)
        print(f"[{index}/{total_manual_links}] Успешно: {current_recipe['name']}")

    except Exception as e:
        print(f"[{index}/{total_manual_links}] Ошибка на {full_url}: {e}")

df_manual = pd.DataFrame(manually_added_recipes_data)

eda_df = pd.read_csv('eda_full_dataset.csv', encoding='utf-16')

combined_df = pd.concat([eda_df, df_manual], ignore_index=True)

combined_df.to_csv('eda_full_dataset.csv', index=False, encoding='utf-16')

print(f"\nСбор данных из вручную добавленных ссылок завершен.")
print(f"Всего собрано: {len(df_manual)} новых рецептов.")
print(f"Общее количество рецептов в eda_full_dataset.csv: {len(combined_df)}")
display(combined_df.head())

In [ ]:
combined_df['description'].isnull().sum()

# DATA CLEANING AND PREPROCESSING

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('eda_full_dataset (1).csv', encoding='utf-16', engine='python', on_bad_lines='warn')
df.shape

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df['cuisine'] = df['cuisine'].fillna("Неизвестно")
df['description'] = df['description'].fillna(df['nlp_text'].str.split('.').str[:3].str.join('. '))
df = df.drop_duplicates()


In [ ]:
df.columns = ['name', 'description','text', 'ingredients', 'portions', 'likes', 'dislikes', 'saved', 'calories', 'protein', 'fat', 'carbs', 'duration', 'cuisine', 'url']
df = df[['name', 'description', 'ingredients', 'portions', 'likes', 'dislikes', 'saved', 'calories', 'protein', 'fat', 'carbs', 'duration', 'cuisine', 'text', 'url']]
df.columns = [col.capitalize() for col in df.columns]

In [ ]:
df['Cuisine'] = df['Cuisine'].astype('category')
df['Ingredients'] = df['Ingredients'].str.split(',')

In [ ]:
df

HANDLING OUTLIERS

In [ ]:
df_outliers = df[['Carbs','Fat','Protein']]
df_outliers.sort_values(by='Protein', ascending=False)
df_outliers.sort_values(by='Fat', ascending=False)
df_outliers.sort_values(by='Carbs', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

num_cols = ['Protein', 'Fat', 'Carbs']

df[num_cols].boxplot()
plt.xticks(rotation=45)
plt.title("Boxplot for Outliers")
plt.show()

After examining the extreme values, no impossible observations were found (e.g., carbohydrates exceeding 600g). The maximum values for Protein, Fat, and Carbs fall within realistic nutritional ranges. Therefore, the detected outliers are likely valid observations rather than data errors, and no outlier treatment is applied.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.boxplot(y=df['Calories'])
plt.title('Boxplot of Calories')
plt.ylabel('Calories')

plt.subplot(1, 3, 2)
sns.boxplot(y=df['Duration'])
plt.title('Boxplot of Duration')
plt.ylabel('Duration')

plt.subplot(1, 3, 3)
sns.boxplot(y=df['Portions'])
plt.title('Boxplot of Portions')
plt.ylabel('Portions')

plt.tight_layout()
plt.show()

In [ ]:

df[df['Duration'] > 2800]

Here for 'Маринованная свекла' is realistic to have that amount of duration to make that dish

In [ ]:
df[df['Calories'] > 2000]

Both this dishes contain such ingridients as butter,cream,sugar and it's realistic to have

In [ ]:
df.loc[df['Name'] == 'Котлетки из брокколи и цветной капусты', 'Ingredients'] = 'Капуста брокколи (450 г), Цветная капуста (450 г), Манная крупа (50 г), Пшеничная мука (40 г), Куриное яйцо (2 штука), Молотый черный перец (0 по вкусу), Черный перец с лимонной цедрой (0 по вкусу), Соль (0 по вкусу), Растительное масло (10 г)'

df.loc[df['Name'] == 'Котлетки из брокколи и цветной капусты', 'Calories'] = 813
df.loc[df['Name'] == 'Котлетки из брокколи и цветной капусты', 'Protein'] = 43
df.loc[df['Name'] == 'Котлетки из брокколи и цветной капусты', 'Fat'] = 23
df.loc[df['Name'] == 'Котлетки из брокколи и цветной капусты', 'Carbs'] = 120


df.loc[df['Name'] == 'Александрийский кулич', 'Ingredients'] = 'Сливочное масло (180 г), Куриное яйцо (4 штука), Яичный желток (2 штука), Топленое молоко (370 мл), Свежие дрожжи (45 г), Светлый изюм (100 г), Темный изюм (100 г), Вяленая клюква (150 г), Коньяк (30 мл), Шафран (0 щепотка), Ванильный сахар (15 г), Цедра апельсина (10 г), Мускатный орех (0 по вкусу), Кардамон (0.5 чайная ложка), Соль (0.5 чайная ложка), Пшеничная мука (1000 г), Яичный белок (2 штука), Сахарная пудра (200 г)'

df.loc[df['Name'] == 'Александрийский кулич', 'Calories'] = 640
df.loc[df['Name'] == 'Александрийский кулич', 'Protein'] = 12.5
df.loc[df['Name'] == 'Александрийский кулич', 'Fat'] = 15.8
df.loc[df['Name'] == 'Александрийский кулич', 'Carbs'] = 103.3


df.loc[df['Name'] == 'Мясо по-испански', 'Ingredients'] = 'Свиная корейка (800 г), Шампиньоны (500 г), Помидоры (300 г), Чеснок (20 г), Сливки (200 мл), Соль (0 по вкусу), Молотый черный перец (0 по вкусу), Сыр (100 г)'

df.loc[df['Name'] == 'Мясо по-испански', 'Calories'] = 781
df.loc[df['Name'] == 'Мясо по-испански', 'Protein'] = 50
df.loc[df['Name'] == 'Мясо по-испански', 'Fat'] = 60
df.loc[df['Name'] == 'Мясо по-испански', 'Carbs'] = 8.5


df.loc[df['Name'] == 'Гаспачо со страчателлой и клубникой', 'Ingredients'] = 'Сладкий перец (150 г), Томатный сок (500 мл), Стебель сельдерея (60 г), Красный лук (70 г), Листья базилика (8 г), Огурцы (170 г), Помидоры (600 г), Перец чили (2 г), Белый винный уксус (25 мл), Вустерширский соус (15 г), Соль (10 г), Сахар (5 г), Чеснок (80 г), Белый хлеб (100 г), Сыр страчателла (50 г), Клубника (50 г), Оливковое масло (2 мл), Бальзамик малиновый (2 г)'

df.loc[df['Name'] == 'Гаспачо со страчателлой и клубникой', 'Calories'] = 237
df.loc[df['Name'] == 'Гаспачо со страчателлой и клубникой', 'Protein'] = 5
df.loc[df['Name'] == 'Гаспачо со страчателлой и клубникой', 'Fat'] = 3.8
df.loc[df['Name'] == 'Гаспачо со страчателлой и клубникой', 'Carbs'] = 42.5


df.loc[df['Name'] == 'Ассорти «Фестиваль шашлыка»', 'Ingredients'] = 'Говядина (200 г), Жир (40 г), Репчатый лук (40 г), Свиная шея (250 г), Куриная грудка (250 г), Баранья корейка (600 г), Мини-картофель (450 г), Шампиньоны (90 г), Красный лук (120 г), Баклажаны (300 г), Сладкий перец (150 г), Помидоры (450 г), Зелень (0 по вкусу), Соль (0 по вкусу), Молотый черный перец (0 по вкусу), Молотый красный перец (0 по вкусу)'

df.loc[df['Name'] == 'Ассорти «Фестиваль шашлыка»', 'Calories'] = 643
df.loc[df['Name'] == 'Ассорти «Фестиваль шашлыка»', 'Protein'] = 40
df.loc[df['Name'] == 'Ассорти «Фестиваль шашлыка»', 'Fat'] = 47
df.loc[df['Name'] == 'Ассорти «Фестиваль шашлыка»', 'Carbs'] = 20


df.loc[df['Name'] == 'Тосты с авокадо и икрой', 'Ingredients'] = 'Ржаной хлеб для тостов (300 г), Авокадо (300 г), Репчатый лук (60 г), Помидоры (150 г), Лайм (70 г), Паприка (0 по вкусу), Чеснок (5 г), Кинза (10 г), Оливковое масло (20 г), Икра (150 г)'

df.loc[df['Name'] == 'Тосты с авокадо и икрой', 'Calories'] = 445
df.loc[df['Name'] == 'Тосты с авокадо и икрой', 'Protein'] = 14.5
df.loc[df['Name'] == 'Тосты с авокадо и икрой', 'Fat'] = 17.5
df.loc[df['Name'] == 'Тосты с авокадо и икрой', 'Carbs'] = 35



df.loc[df['Name'] == 'Пельмени из оленины со сметаной', 'Ingredients'] = 'Оленина (300 г), Филе куриного бедра (100 г), Розмарин (15 г), Репчатый лук (200 г), Сливки 33%-ные (80 мл), Куриное яйцо (2 штука), Пшеничная мука (600 г), Вода (300 мл), Соль (6 г), Растительное масло (20 мл), Куриный бульон (0 по вкусу)'

df.loc[df['Name'] == 'Пельмени из оленины со сметаной', 'Calories'] = 3455
df.loc[df['Name'] == 'Пельмени из оленины со сметаной', 'Protein'] = 154
df.loc[df['Name'] == 'Пельмени из оленины со сметаной', 'Fat'] = 94
df.loc[df['Name'] == 'Пельмени из оленины со сметаной', 'Carbs'] = 450



df.loc[df['Name'] == 'Цветная капуста под шубой из рикотты', 'Ingredients'] = 'Чеснок (1 головка), Розмарин (2 веточки), Оливковое масло (70 мл), Цветная капуста (1 штука), Белое сухое вино (250 мл), Бульон (350 мл), Лимон (1 штука), Сливочное масло (50 г), Перец чили хлопьями (1 столовая ложка), Лавровый лист (2 штуки), Мускатный орех (0 щепотка), Тертый сыр пармезан (120 г), Сыр рикотта (200 г), Цельное молоко (75 мл), Копченый бекон (30 г), Соль (0 по вкусу), Молотый черный перец (0 по вкусу)'
df.loc[df['Name'] == 'Цветная капуста под шубой из рикотты', 'Calories'] = 620
df.loc[df['Name'] == 'Цветная капуста под шубой из рикотты', 'Protein'] = 17.5
df.loc[df['Name'] == 'Цветная капуста под шубой из рикотты', 'Fat'] = 42.5
df.loc[df['Name'] == 'Цветная капуста под шубой из рикотты', 'Carbs'] = 30

In [ ]:
mask = df['Calories'] > 1600

df.loc[mask, 'Calories'] = df['Calories'] / df['Portions']
df.loc[mask, 'Protein'] = df['Protein'] / df['Portions']
df.loc[mask, 'Fat'] = df['Fat'] / df['Portions']
df.loc[mask, 'Carbs'] = df['Carbs'] / df['Portions']

In [ ]:
df.to_csv('clean_recipe_dataset.csv',index = False)


# DATA TRANSFORMATION & FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('clean_recipe_dataset.csv')

Label Encoding

In [ ]:
le = LabelEncoder()
df['cuisine_encoded'] = le.fit_transform(df['Cuisine'])
print(f"  Example: 'Европейская кухня' → {le.transform(['Европейская кухня'])[0]}")
result = pd.DataFrame({
    'cuisine': le.classes_,
    'cuisine_encoded': le.transform(le.classes_)
})

print(result.head(10).to_string(index=False))

Feature Engineering

In [ ]:
import pandas as pd

def categorize_nutrition(row):
    if row['Calories'] < 300:
        return 'Low Calorie'
    if row['Protein'] > (row['Fat'] + row['Carbs']):
        return 'High Protein'

    macros = {
        'High Carbs': row['Carbs'],
        'High Fat': row['Fat'],
        'Balanced': row['Protein']
    }

    dominant_macro = max(macros, key=macros.get)

    if abs(row['Carbs'] - row['Fat']) < 5:
        return 'Balanced'

    return dominant_macro

df['Dietary_Focus'] = df.apply(categorize_nutrition, axis=1)


df[['Name', 'Calories', 'Protein', 'Fat', 'Carbs', 'Dietary_Focus']].head()

In [ ]:
df.to_csv('recipe_links.csv')

In [ ]:
links = pd.read_csv('recipe_links.csv')

In [ ]:
df['Dish_Type'] = links['Url'].str.split('/').str[1]

In [ ]:
dish_type_mapping = {
    'osnovnye-blyuda': 'main-courses',
    'vypechka-deserty': 'baking-desserts',
    'zakuski': 'appetizers',
    'salaty': 'salads',
    'supy': 'soups',
    'napitki': 'drinks',
    'zavtraki': 'breakfasts',
    'pasta-picca': 'pasta-pizza',
    'sousy-marinady': 'sauces-marinades',
    'zagotovki': 'preparations',
    'sendvichi': 'sandwiches',
    'rizotto': 'risotto',
    'bulony': 'broths'
}

df['Dish_Type'] = df['Dish_Type'].replace(dish_type_mapping)
df['Dish_Type'].value_counts()

'Calories-per-portion'feature

In [ ]:
df['calories_per_portion'] = np.where(
    df['Portions'] > 0,
    (df['Calories'] / df['Portions']).round(2), 0.0)
print(f"calories_per_portion: mean={df['calories_per_portion'].mean():.1f}")
df[['Name', 'calories_per_portion']].head()

Time_Effort categorical future about duration of meal preparation

In [ ]:
df['Time_Effort'] = pd.cut(df['Duration'],
                           bins=[0, 20, 60, 1000],
                           labels=['Express', 'Standard', 'Time-Consuming'])
df['Time_Effort']

Using df exploded to  further visualization

In [ ]:
df['Ingredients_List'] = df['Ingredients'].str.replace(r'[\[\]]', '', regex=True)

df['Ingredients_List'] = df['Ingredients_List'].str.split(',')

df_exploded = df.explode('Ingredients_List')

df_exploded['Ingredient_Final'] = (
    df_exploded['Ingredients_List']
    .str.replace(r'\(.*\)', '', regex=True) # Removes
    .str.strip()                             # Removes extra spaces
    .str.lower()                             # Normalizes to lowercase
)
df_exploded = df_exploded[['Name', 'Ingredient_Final', 'Portions', 'Calories', 'Protein', 'Fat', 'Carbs', 'Likes', 'Dislikes', 'Saved', 'Duration', 'Cuisine', 'Dietary_Focus', 'Dish_Type']]
df_exploded.shape

In [ ]:
df_exploded.to_csv('exploded_ingredients.csv', index = False)

In [ ]:
df_exploded

Standartization for further ML

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_features = ['Calories', 'Protein', 'Fat', 'Duration', 'Portions', 'Likes', 'Dislikes', 'Saved']

# 1. Create the Scaler
scaler_std = StandardScaler()

# 2. Define names for the new scaled columns
std_cols = [f'{c}_std' for c in numeric_features]

# 3. Fit AND assign (Crucial step!)
df[std_cols] = scaler_std.fit_transform(df[numeric_features])

print("Columns available for Visualisation:", numeric_features)
print("Columns available for Machine Learning:", std_cols)

In [ ]:
df


In [ ]:
df.to_csv('Glaze_of_glory_dataset.csv',index = False)

In [ ]:
df_exploded.to_csv('exploded_ingredients.csv')

# EDA & VISUALIZATION

In [ ]:
df = pd.read_csv('Glaze_of_glory_dataset.csv')

In [ ]:
df_eda = df.copy()
df_eda = df_eda.drop(columns = ['Ingredients_List',
       'Calories_std', 'Protein_std', 'Fat_std', 'Duration_std',
       'Portions_std', 'Likes_std', 'Dislikes_std', 'Saved_std'])
df_eda

In [ ]:
df_eda['len'] = df_eda['Ingredients'].str.len()
df_eda['len'].describe()

In [ ]:
df_eda.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


df_cuisine = df_eda['Cuisine'].value_counts()
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 15))
axes = axes.flatten()

# distribution of the number of recipes per cuisine
sns.histplot(df_cuisine, kde=True, bins=20, edgecolor='black', ax=axes[0])
axes[0].set_title('Distribution of Recipe Counts per Cuisine')
axes[0].set_xlabel('Number of Recipes')
axes[0].set_ylabel('Number of Cuisines')

# Distribution of Duration
sns.histplot(df_eda['Duration'], kde=True, bins=20, edgecolor='black', ax=axes[1])
axes[1].set_title('Distribution of Duration')
axes[1].set_xlabel('Duration')
axes[1].set_ylabel('Frequency')

# Distribution of Calories
sns.histplot(df_eda['Calories'], kde=True, bins=20, edgecolor='black', ax=axes[2])
axes[2].set_title('Distribution of Calories')
axes[2].set_xlabel('Calories')
axes[2].set_ylabel('Frequency')

# Distribution of Protein
sns.histplot(df_eda['Protein'], kde=True, bins=20, edgecolor='black', ax=axes[3])
axes[3].set_title('Distribution of Protein')
axes[3].set_xlabel('Protein')
axes[3].set_ylabel('Frequency')

# Distribution of Fat
sns.histplot(df_eda['Fat'], kde=True, bins=20, edgecolor='black', ax=axes[4])
axes[4].set_title('Distribution of Fat')
axes[4].set_xlabel('Fat')
axes[4].set_ylabel('Frequency')

# Distribution of Carbs
sns.histplot(df_eda['Carbs'], kde=True, bins=20, edgecolor='black', ax=axes[5])
axes[5].set_title('Distribution of Carbs')
axes[5].set_xlabel('Carbs')
axes[5].set_ylabel('Frequency')

# Distribution of Likes
sns.histplot(df_eda['Likes'], kde=True, bins=20, edgecolor='black', ax=axes[6])
axes[6].set_title('Distribution of Likes')
axes[6].set_xlabel('Likes')
axes[6].set_ylabel('Frequency')

# Distribution of Dislikes
sns.histplot(df_eda['Dislikes'], kde=True, bins=20, edgecolor='black', ax=axes[7])
axes[7].set_title('Distribution of Dislikes')
axes[7].set_xlabel('Dislikes')
axes[7].set_ylabel('Frequency')

# Distribution of Saved
sns.histplot(df_eda['Saved'], kde=True, bins=20, edgecolor='black', ax=axes[8])
axes[8].set_title('Distribution of Saved')
axes[8].set_xlabel('Saved')
axes[8].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
avg_calories_per_cuisine = df_eda.groupby('Cuisine')['Calories'].mean().sort_values(ascending=False)
avg_protein_per_cuisine_unsorted = df_eda.groupby('Cuisine')['Protein'].mean()
avg_protein_per_cuisine = avg_protein_per_cuisine_unsorted.reindex(avg_calories_per_cuisine.index)
avg_fat_per_cuisine = df_eda.groupby('Cuisine')['Fat'].mean()
avg_fat_per_cuisine = avg_fat_per_cuisine.reindex(avg_calories_per_cuisine.index)
avg_carbs_per_cuisine = df_eda.groupby('Cuisine')['Carbs'].mean()
avg_carbs_per_cuisine = avg_carbs_per_cuisine.reindex(avg_calories_per_cuisine.index)
avg_duration_per_cuisine = df_eda.groupby('Cuisine')['Duration'].mean().sort_values(ascending=False)
count_recipes_per_cuisine = df_eda['Cuisine'].value_counts()
count_recipes_per_cuisine = count_recipes_per_cuisine.reindex(avg_calories_per_cuisine.index)


In [ ]:
count_recipes_per_cuisine = df_eda['Cuisine'].value_counts()
count_recipes_per_cuisine = count_recipes_per_cuisine.reindex(avg_calories_per_cuisine.index)

plt.figure(figsize=(20, 6))
count_recipes_per_cuisine.plot(kind='bar', color='lightcoral', edgecolor='black')
plt.title('Recipes by Cuisine (Sorted by Calories)', fontsize=14)
plt.xlabel('Cuisine', fontsize=12)
plt.ylabel('Recipe Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 6))
avg_calories_per_cuisine.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Average Calories by Cuisine (Sorted by Calories)', fontsize=14)
plt.xlabel('Cuisine', fontsize=12)
plt.ylabel('Average Calories', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 6))
avg_protein_per_cuisine.plot(kind='bar', color='lightcoral', edgecolor='black')
plt.title('Average Protein by Cuisine (Sorted by Calories)', fontsize=14)
plt.xlabel('Cuisine', fontsize=12)
plt.ylabel('Average Protein', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 6))
avg_fat_per_cuisine.plot(kind='bar', color='lightgreen', edgecolor='black')
plt.title('Average Fat by Cuisine (Sorted by Calories)', fontsize=14)
plt.xlabel('Cuisine', fontsize=12)
plt.ylabel('Average Fat', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 6))
avg_carbs_per_cuisine.plot(kind='bar', color='lightcoral', edgecolor='black')
plt.title('Average Carbs by Cuisine (Sorted by Calories)', fontsize=14)
plt.xlabel('Cuisine', fontsize=12)
plt.ylabel('Average Carbs', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

All distributions are sorted in descending order by calories. We can observe that the cuisines with the highest calorie values, as well as those with other extreme characteristics, tend to be the ones with the smallest number of dishes in our dataset. Therefore, we cannot conclude that these cuisines are inherently high-calorie or low-calorie, since the analysis in these cases is based on a very limited sample size. In contrast, the cuisines with the largest number of dishes in the dataset tend to appear in the middle of the distribution, which is expected and statistically more reliable.

The distribution of calories is very similar to the distribution of fat. From this, we can infer that fat has the strongest influence on total calorie content, followed by carbohydrates, and then protein, based on the similarity of their respective distributions.

In [ ]:
correlation_matrix = df_eda.corr(numeric_only = True)
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

Most of columns are not correlated, however there is unexpected correlation between fat and protein;

correlation between calories, proteins, fat, carbs is same as we guessed at previous distributions

---



In [ ]:
df_eda[df_eda['Saved'] > 1000][['Saved','Cuisine', 'Time_Effort']]

In [ ]:
import matplotlib.pyplot as plt

filtered_df = df_eda[df_eda['Saved'] > 1000][['Saved','Cuisine', 'Time_Effort']]

cuisine_counts = filtered_df['Cuisine'].value_counts()

threshold = 0.015
cuisine_percentages = cuisine_counts / cuisine_counts.sum()


cuisine_counts_other = cuisine_counts[cuisine_percentages >= threshold]
cuisine_counts_other = pd.concat([cuisine_counts_other, pd.Series([cuisine_counts[cuisine_percentages < threshold].sum()], index=['Other'])])

plt.figure(figsize=(10, 6))
plt.pie(cuisine_counts_other, labels=cuisine_counts_other.index, autopct='%1.1f%%', startangle=240)
plt.title('Cuisine Distribution for Recipes with High Saves')
plt.axis('equal')
plt.show()


duration_counts = filtered_df['Time_Effort'].value_counts()

plt.figure(figsize=(10, 6))
plt.pie(duration_counts, labels=duration_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Duration Category Distribution for Recipes with High Saves')
plt.axis('equal')
plt.show()

In [ ]:
import plotly.express as px

# Calculate cuisine frequencies
cuisine_counts = df['Cuisine'].value_counts()

# Get the 10 most frequent cuisines
top_10_cuisines = cuisine_counts.head(10).index.tolist()

# Combine the lists of cuisines (in this case, just the top 10)
selected_cuisines = top_10_cuisines

# Filter the DataFrame to include only these selected cuisines
df_filtered = df[df['Cuisine'].isin(selected_cuisines)].copy()

# Create the violin plot using the filtered DataFrame
fig = px.violin(df_filtered, y="Calories", x="Cuisine", box=True, points=None,
                title="Calorie Distribution for 10 Most Popular Cuisines")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

##VISUALIZATION

In [ ]:
df_exploded = pd.read_csv('exploded_ingredients.csv')
ingredients = df_exploded.drop(columns = ['Unnamed:0'], errors='ignore')

In [ ]:
import pandas as pd
df = pd.read_csv('Glaze_of_glory_dataset.csv')
df = df.drop(columns = ['Ingredients_List',
       'Calories_std', 'Protein_std', 'Fat_std', 'Duration_std',
       'Portions_std', 'Likes_std', 'Dislikes_std', 'Saved_std'])

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

###**1.** *The "Quick & Healthy" Paradox*

The Question: Do "Low Calorie" or "High Protein" meals actually take less time to cook than "High Carbs" or "High Fat" meals?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

heatmap_data1 = pd.crosstab(df['Dietary_Focus'], df['Time_Effort'])
sns.heatmap(heatmap_data1, annot=True, fmt="d", cmap="YlGnBu", ax=axes[0, 0], cbar_kws={'label': 'Number of Recipes'})
axes[0, 0].set_title('Nutrition vs. Time Effort', fontsize=13, pad=15)
axes[0, 0].set_xlabel('Preparation Speed')
axes[0, 0].set_ylabel('Nutritional Focus')


heatmap_data2 = pd.crosstab(df['Dish_Type'], df['Time_Effort'])
sns.heatmap(heatmap_data2, annot=True, fmt="d", cmap="YlGnBu", ax=axes[0, 1], cbar_kws={'label': 'Number of Recipes'})
axes[0, 1].set_title('Dish Type vs. Time Effort', fontsize=13, pad=15)
axes[0, 1].set_xlabel('Preparation Speed')
axes[0, 1].set_ylabel('Dish Types')

fig.delaxes(axes[1, 0])
fig.delaxes(axes[1, 1])

ax3 = fig.add_subplot(2, 1, 2)
heatmap_data3 = pd.crosstab(df['Dietary_Focus'], df['Dish_Type'])
percentages = (heatmap_data3.div(heatmap_data3.sum(axis=0), axis=1) * 100).round(1)

sns.heatmap(percentages, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax3, cbar_kws={'label': 'Percentage (%)'})


for t in ax3.texts:
    if "%" not in t.get_text():
        t.set_text(t.get_text() + "%")

ax3.set_title('Dietary Focus Distribution by Dish Type (Proportional)', fontsize=15, pad=20)
ax3.set_xlabel('Dish Type')
ax3.set_ylabel('Nutritional Focus')


plt.subplots_adjust(hspace=0.4, wspace=0.6)

plt.show()

1.Our assumption was true healthy meal types like the ones with low calorie or with high proteins are quicker to prepare than the ones with high carbs and fats.Thats actually good news for people who wants to eat conciously it wiil not take much to take care about your health based on our data

2.
3.If you want to eat something balanced, then choose any main-course or soup, so probability that chosen dish is balanced will be higher than if you choose any other type of dish

### 2. The Law of Diminishing Returns in Ingredients

In [ ]:
ing_count = df_exploded.groupby('Name')['Ingredient_Final'].count().reset_index()
merged = pd.merge(ing_count, df[['Name', 'Saved']], on='Name', how='left')

plt.figure(figsize=(10, 6))
sns.lineplot(data=merged, x='Ingredient_Final', y='Saved', estimator='mean')
plt.title('Dependence of Saves on the Number of Ingredients')
plt.xlabel('Number of Ingredients')
plt.ylabel('Saves')
plt.show()

df['Name_Len'] = df['Name'].apply(len)

fig = px.scatter(df, x='Name_Len', y='Saved', trendline="ols",
                 title='Impact of Title Length on Number of Saves')
fig.show()

For average person for everyday meal ingredients beyond approximately eight unique components, the popularity of a recipe begins to decline.But we can see unexpected growth on 35 which might represent dishes for special events which everybody needs once in a life.While users value variety, excessively long ingredient lists introduce perceived complexity and effort, which may discourage engagement and reduce the likelihood of saving the recipe.
**Insight for graph 2:**
Recipes with short, concise titles (e.g., “Brownies”) are saved more frequently than those with longer, more descriptive names (e.g., “Homemade syrniki made from farm-style cottage cheese”). Simplicity in naming appears to be a key driver of virality.

### **3.** Top 20 Most Frequent Ingredients

In [ ]:
import pandas as pd

# Using the df_exploded and df DataFrames already available in the notebook's environment
# df_exploded contains the 'Ingredient_Final' column which was pre-processed
# df contains the 'Likes' column

global_avg_likes = df['Likes'].mean()

ing_stats = df_exploded.groupby('Ingredient_Final').agg(
    count=('Likes', 'count'),
    avg_likes=('Likes', 'mean')
).reset_index()

min_occurrence = 10
ing_stats = ing_stats[ing_stats['count'] > min_occurrence]

ing_stats = ing_stats.sort_values('count', ascending=False).iloc[15:]

ing_stats['lift_%'] = ((ing_stats['avg_likes'] - global_avg_likes) / global_avg_likes) * 100


top_drivers = ing_stats.sort_values('lift_%', ascending=False).head(10)
top_drivers[['Ingredient_Final', 'count', 'lift_%']]

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Calculate the 'Lift' from your ingredient stats
# Assuming 'ing_stats' is the dataframe from the previous step
# 'lift_%' = ((Avg Likes of Ingredient - Global Avg) / Global Avg) * 100

# Let's take the Top 10 Drivers and Bottom 10 Killers for a clean story
top_10 = ing_stats.sort_values('lift_%', ascending=False).head(10)
bottom_10 = ing_stats.sort_values('lift_%', ascending=True).head(10)
plot_df = pd.concat([top_10, bottom_10]).sort_values('lift_%')

# 2. Create the Diverging Bar Chart
fig = px.bar(
    plot_df,
    x='lift_%',
    y='Ingredient_Final',
    orientation='h',
    title='Ingredient Impact: Popularity Drivers vs. Engagement Killers',
    labels={'lift_%': 'Popularity Lift/Drag (%)', 'Ingredient_Final': 'Ingredient'},
    color='lift_%',
    color_continuous_scale=[[0, '#e74c3c'], [0.5, '#f1c40f'], [1, '#27ae60']],
    color_continuous_midpoint=0
)

# 3. Formatting for a professional look
fig.update_layout(
    template='plotly_white',
    height=600,
    xaxis_title="Impact on Likes (%)",
    yaxis_title="",
    coloraxis_showscale=False # Clean look
)

# Add a vertical line at zero
fig.add_vline(x=0, line_width=2, line_color="black")

fig.show()

Certain ingredients are highly prevalent across the dataset, indicating that they serve as fundamental components in a wide variety of recipes. Their frequent usage highlights their role as essential building blocks in culinary practices.

### **4.Which Cuisines Are Most Loved**

In [ ]:
df.columns

In [ ]:
df['Like_Ratio'] = df['Likes']/(df['Likes']+df['Dislikes'])

In [ ]:
import plotly.express as px

cuisine_stats = df.groupby('Cuisine').agg({'Likes': 'mean', 'Name': 'count'}).reset_index()
cuisine_stats.columns = ['Cuisine', 'avg_likes', 'recipe_count']

# Filter for cuisines with enough recipes to be relevant (e.g., more than 10)
popular_cuisines = cuisine_stats[cuisine_stats['recipe_count'] > 10].sort_values(by='avg_likes', ascending=False)

fig1 = px.bar(
    popular_cuisines.head(10),
    x='avg_likes',
    y='Cuisine',
    orientation='h',
    title='Top 10 Most Popular Cuisines (Average Likes)',
    color='avg_likes',
    color_continuous_scale='viridis',
    labels={'avg_likes': 'Average Likes', 'Cuisine': 'Country/Cuisine'}
)
fig1.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure df_eda is available
df = pd.read_csv('Glaze_of_glory_dataset.csv')
df_eda = df.copy()
df_eda = df_eda.drop(columns = ['Ingredients_List',
       'Calories_std', 'Protein_std', 'Fat_std', 'Duration_std',
       'Portions_std', 'Likes_std', 'Dislikes_std', 'Saved_std'], errors='ignore')

# Calculate the number of ingredients for each recipe
df_eda['num_ingredients'] = df_eda['Ingredients'].apply(lambda x: len(x.strip("[]").split(',')) if pd.notna(x) and x.strip("[]") != '' else 0)

# List of desired cuisines (Russian names for filtering)
selected_cuisines_russian = ['Испанская кухня', 'Украинская кухня', 'Белорусская кухня', 'Итальянская кухня', 'Французская кухня']

# Mapping from Russian to English names for plotting
cuisine_name_mapping = {
    'Испанская кухня': 'Spanish Cuisine',
    'Украинская кухня': 'Ukrainian Cuisine',
    'Белорусская кухня': 'Belarusian Cuisine',
    'Итальянская кухня': 'Italian Cuisine',
    'Французская кухня': 'French Cuisine'
}

# Filter the DataFrame for these cuisines
df_filtered_cuisines = df_eda[df_eda['Cuisine'].isin(selected_cuisines_russian)]

# Calculate the average number of ingredients for each selected cuisine
avg_ingredients_per_cuisine = df_filtered_cuisines.groupby('Cuisine')['num_ingredients'].mean().sort_values(ascending=False)

# Map Russian index names to English for the plot
avg_ingredients_per_cuisine.index = avg_ingredients_per_cuisine.index.map(cuisine_name_mapping)

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_ingredients_per_cuisine.index, y=avg_ingredients_per_cuisine.values, palette='viridis')
plt.title('Average Number of Ingredients for Selected Cuisines')
plt.xlabel('Cuisine')
plt.ylabel('Average Number of Ingredients')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### **5.Calory multipliers**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Custom styling
# Removed: plt.style.use('dark_background')
df = pd.read_csv('exploded_ingredients.csv')

# Calculate average calories per ingredient presence
ing_stats = df.groupby('Ingredient_Final')['Calories'].agg(['mean', 'count'])
ing_stats = ing_stats[ing_stats['count'] > 5].sort_values('mean', ascending=False).head(15)
ing_stats.index = ing_stats.index.str.replace("'", "").str.strip().str.title()

plt.figure(figsize=(12, 8))
colors = sns.color_palette("pastel", len(ing_stats))
barplot = sns.barplot(x='mean', y=ing_stats.index, data=ing_stats, palette=colors, edgecolor='black')


plt.axvline(df['Calories'].mean(), color='gray', linestyle='--', label=f"Global Avg: {df['Calories'].mean():.0f}")

plt.title('Caloric Heavyweights: Mean Recipe Calories when ingredient is present', fontsize=16, pad=20)
plt.xlabel('Average Calories of Containing Recipes', fontsize=12)
plt.ylabel('Ingredient', fontsize=12)
plt.grid(axis='x', alpha=0.2)
plt.legend()
sns.despine()
plt.show()

We identified ingredients that significantly increase the average caloric content of recipes they are present in, such as 'Булочка С Кунжутом' (bun with sesame), 'Шпик' (pork fat), and 'Свиные Ребра' (pork ribs). This insight can be crucial for users looking to manage their caloric intake or for developing recommendations for specific dietary needs.

## **6.Which ones are most controversial recipes?**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('default')

df_exploded = pd.read_csv('exploded_ingredients.csv')
df_exploded['Ingredient_Final'] = df_exploded['Ingredient_Final'].str.replace("'", "").str.strip()

recipe_ingredient_counts = df_exploded.groupby('Name')['Ingredient_Final'].count().reset_index()
recipe_ingredient_counts.rename(columns={'Ingredient_Final': 'Num_Ingredients_Per_Recipe'}, inplace=True)

unique_recipes_metadata = df_exploded.drop_duplicates(subset=['Name']).copy()
unique_recipes_metadata = unique_recipes_metadata[['Name', 'Likes', 'Dislikes', 'Duration', 'Dish_Type']]

recipe_stats = pd.merge(unique_recipes_metadata, recipe_ingredient_counts, on='Name', how='left')


popular_recipes = recipe_stats[recipe_stats['Likes'] > 100].copy()
popular_recipes['Controversy_Index'] = popular_recipes['Dislikes'] / popular_recipes['Likes']


controversial_dish_type_stats = popular_recipes.groupby('Dish_Type').agg(
    Avg_Controversy_Index=('Controversy_Index', 'mean'),
    Avg_Duration=('Duration', 'mean'),
    Avg_Num_Ingredients=('Num_Ingredients_Per_Recipe', 'mean')
).reset_index()

# Sort by average controversy index
top_controversial = controversial_dish_type_stats.sort_values('Avg_Controversy_Index', ascending=False).head(15)

norm = plt.Normalize(top_controversial['Avg_Duration'].min(), top_controversial['Avg_Duration'].max())
colors = plt.cm.coolwarm(norm(top_controversial['Avg_Duration']))

plt.figure(figsize=(12, 8))

plt.barh(
    top_controversial['Dish_Type'],
    top_controversial['Avg_Controversy_Index'],
    color=colors,
    edgecolor='gray'
)

plt.title('Which Of These Dish Types Are Controversial? (Average Metrics)', fontsize=16)
plt.xlabel('Average Dislikes per 1 Like', fontsize=12)

for i, (val, ing_avg) in enumerate(zip(top_controversial['Avg_Controversy_Index'],
                                   top_controversial['Avg_Num_Ingredients'])):
    plt.text(val, i, f"{ing_avg:.1f} avg ing", va='center')

sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=plt.gca(), label='Average Cooking Time (minutes)')

plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

sns.despine()
plt.tight_layout()
plt.show()

**Insight:**
The cooking times across these recipes vary significantly, suggesting that preparation time is unlikely to be the primary driver of negative feedback. Similarly, the number of ingredients differs across recipes, indicating no clear pattern in terms of complexity.
**Insight2:**
It was unexpected to find that breakfast and beverage categories exhibit the highest levels of controversy compared to other dish types. Although these recipes are generally perceived as simple and easy to prepare, they show disproportionately high levels of user disagreement. This suggests that, unlike other categories, dissatisfaction in these cases is less related to complexity and more likely driven by unmet expectations in taste, quality, or perceived value.


The distribution of recipes across dietary categories shows that 'High Carbs' and 'Low Calorie' recipes are the most prevalent, followed by 'High Fat', 'Balanced', and 'High Protein' recipes. This highlights the diversity of culinary focus within the dataset.

In [ ]:
df.columns

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

unhealthy_list = ['High Fat', 'High Carbs']
df['Health_Category'] = df['Dietary_Focus'].apply(
    lambda x: 'Indulgent' if x in unhealthy_list else 'Health-Conscious'
)

# Calculate maximum likes for each Dietary_Focus
max_likes_per_focus = df.groupby('Dietary_Focus')['Likes'].max().reset_index().sort_values(by='Likes', ascending=False)

fig1 = px.bar(
    max_likes_per_focus,
    x='Dietary_Focus',
    y='Likes',
    color='Dietary_Focus',
    title='Maximum Popularity by Dietary Focus',
    labels={'Likes': 'Maximum Likes', 'Dietary_Focus': 'Dietary Category'}
)

fig1.update_traces(texttemplate='%{y:.0f}', textposition='outside')
fig1.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

unhealthy_list = ['High Fat', 'High Carbs']
df['Health_Category'] = df['Dietary_Focus'].apply(
    lambda x: 'Indulgent' if x in unhealthy_list else 'Health-Conscious'
)

focus_stats = df.groupby('Dietary_Focus')['Likes'].mean().reset_index().sort_values(by='Likes', ascending=False)

fig1 = px.bar(
    focus_stats,
    x='Dietary_Focus',
    y='Likes',
    color='Dietary_Focus',
    title='Average Popularity by Dietary Focus',
    labels={'Likes': 'Average Likes', 'Dietary_Focus': 'Dietary Category'}
)

fig1.update_traces(texttemplate='%{y:.2f}', textposition='outside')
fig1.show()


In [ ]:
import plotly.express as px

high_fat_df = df[df['Dietary_Focus'] == 'High Fat']

high_fat_cuisine_stats = high_fat_df.groupby('Cuisine').agg({
    'Likes': ['mean', 'count', 'max']
}).reset_index()

# Flatten columns for easier plotting
high_fat_cuisine_stats.columns = ['Cuisine', 'Avg_Likes', 'Recipe_Count', 'Max_Likes']

# 3. Visualize the "High Fat Landscape"
fig = px.bar(
    high_fat_cuisine_stats.sort_values('Avg_Likes', ascending=False),
    x='Cuisine',
    y='Avg_Likes',
    color='Recipe_Count', # Darker colors = more common in your dataset
    title='Which Cuisines Drive the "High Fat" Category?',
    labels={'Avg_Likes': 'Average Likes', 'Recipe_Count': 'Number of Recipes'}
)
fig.show()

In [ ]:
df.columns

In [ ]:
import pandas as pd

df = pd.read_csv('Glaze_of_glory_dataset.csv')

# Calculate ingredients_count from the 'Ingredients' column
df['ingredients_count'] = df['Ingredients'].apply(lambda x: len([item.strip() for item in str(x).split(',') if item.strip()]) if pd.notna(x) else 0)

correlation = df['ingredients_count'].corr(df['Saved'])
print(f"Correlation between number of ingredients and saves: {correlation:.4f}")

# 2. Investigate the 35+ "Spike"
total_recipes = len(df)
high_ingredient_recipes = df[df['ingredients_count'] >= 35]
count_high = len(high_ingredient_recipes)
percentage_high = (count_high / total_recipes) * 100

print(f"Number of recipes with 35+ ingredients: {count_high}")
print(f"Percentage of total dataset: {percentage_high:.2f}%")
print(f"Average saves for 35+ ingredient recipes: {high_ingredient_recipes['Saved'].mean():.2f}")
print(f"Average saves for the rest: {df[df['ingredients_count'] < 35]['Saved'].mean():.2f}")

# Hypothesis Testing & Time Series Analysis

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df= pd.read_csv("clean_recipe_dataset.csv",index_col=0)
df

## Hypothesis Testing

In [ ]:
import scipy.stats as stats
import pandas as pd



### Hypothesis 1: Is there a significant difference in 'Likes' between 'Русская кухня' and 'Итальянская кухня'?

In [ ]:


russian_cuisine_likes = df[df['Cuisine'] == 'Русская кухня']['Likes']
italian_cuisine_likes = df[df['Cuisine'] == 'Итальянская кухня']['Likes']

t_stat, p_value = stats.ttest_ind(russian_cuisine_likes, italian_cuisine_likes, equal_var=False)

print(f"Hypothesis 1: Russian cuisine vs. Italian cuisine (Like")
print(f"Mean Likes for Русская кухня: {russian_cuisine_likes.mean():.2f}")
print(f"Mean Likes for Итальянская кухня: {italian_cuisine_likes.mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.3f}")

alpha = 0.05
if p_value < alpha:
    print("Result: Reject the null hypothesis. There is a significant difference in average likes.")
else:
    print("Result: Fail to reject the null hypothesis. There is no significant difference in average likes.")

### Hypothesis 2: Do recipes with higher calorie content receive significantly more likes than 'healthy' recipes?


In [ ]:

df['calorie_tier'] = pd.cut(
    df['Calories'],
    bins=[0, 200, 400, 600, float('inf')],
    labels=['low_cal', 'medium_cal', 'high_cal', 'very_high_cal']
)


healthy_recipes_likes = df[df['calorie_tier'] == 'low_cal']['Likes']

high_cal_fat_recipes_likes = df[(df['calorie_tier'] == 'high_cal') | (df['calorie_tier'] == 'very_high_cal')]['Likes']


t_stat, p_value = stats.ttest_ind(healthy_recipes_likes, high_cal_fat_recipes_likes, equal_var=False)

print(f"\n--- Hypothesis 2: Low-Calorie vs. High-Calorie/Fat Recipes (Likes) ---")
print(f"Mean Likes for 'Low-Calorie' recipes: {healthy_recipes_likes.mean():.2f}")
print(f"Mean Likes for 'High-Calorie/Fat' recipes: {high_cal_fat_recipes_likes.mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.3f}")

alpha = 0.05
if p_value < alpha:
    print("Result: Reject the null hypothesis. There is a significant difference in average likes.")
else:
    print("Result: Fail to reject the null hypothesis. There is no significant difference in average likes.")

##Hypothesis 3:
We wanted to check the stereotype: "Diet food is simple." We compared the amount of Carbohydrates (Carbs) to the number of ingredients.

In [ ]:
import pandas as pd
from scipy import stats

df['num_ingredients'] = df['Ingredients'].apply(lambda x: len(str(x).split(',')) if pd.notnull(x) else 0)

rho_fast, p_fast = stats.spearmanr(df['Carbs'], df['Duration'])
rho_simple, p_simple = stats.spearmanr(df['Carbs'], df['num_ingredients'])

print(f"--- Hypothesis Testing (English) ---")
print(f"Is it FAST? Correlation (Carbs/Duration): {rho_fast:.4f} (p-value: {p_fast:.2e})")
print(f"Is it SIMPLE? Correlation (Carbs/Ingredients): {rho_simple:.4f} (p-value: {p_simple:.2e})")

if rho_fast > 0 and p_fast < 0.05:
    print("\nResult: Significant positive correlation found.")
    print("Interpretation: Lower carb (dietary) food tends to have shorter cooking durations.")
else:
    print("\nResult: No significant link between 'dietary' and 'speed'.")

# ML



model to guess the Calories just by looking at the amount of Fat, Protein, and Carbs in the meal.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

features_kbju = ['Protein', 'Fat', 'Carbs']
X = df[features_kbju]
y = df['Calories']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_kbju = RandomForestRegressor(n_estimators=100, random_state=42)
reg_kbju.fit(X_train, y_train)


y_pred = reg_kbju.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Results for Calories Model:")
print(f"RMSE: {rmse:.2f} kcal")
print(f"R²: {r2:.2f}")


importances = reg_kbju.feature_importances_
for name, val in zip(features_kbju, importances):
    print(f"{name}: {val:.2%}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

plt.title('How accurately does the model predict calories?')
plt.xlabel('Actual Calories (from dataset')
plt.ylabel('Predicted Calories')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Advanced ML

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documents = df['Text'].tolist()


embeddings = model.encode(documents, convert_to_tensor=True, show_progress_bar=True)


def find_similar_recipes(query, k=5):
    query_embedding = model.encode(query, convert_to_tensor=True)


    cosine_scores = util.cos_sim(query_embedding, embeddings)[0]


    top_results = torch.topk(cosine_scores, k=k)

    print(f"Search results for: '{query}'\n")

    for score, idx in zip(top_results[0], top_results[1]):
        row_idx = int(idx)
        print(f"Similarity Score: {score:.2f}")
        print(f"Recipe: {documents[row_idx][:150]}...")
        print("-" * 50)

find_similar_recipes("ПП завтрак с яйцами")

In [ ]:
find_similar_recipes("быстрый ужин с курицей")

In [ ]:
find_similar_recipes("завтрак для детей")

In [ ]:
df

In [ ]:
df = df.drop(columns = ['calorie_tier','Text','Url','Ingridients'], errors='ignore')
df.to_csv('Try.csv')

In [ ]:
df.columns

In [ ]:
df1 = pd.read_csv('Glaze_of_glory_dataset.csv')
df1 = df1.drop(columns = ['Text','Url'],errors ='ignore')
df1.columns

In [ ]:
df1.to_csv('try2.csv')